## Merge counts (measurement estimates) and water type 

In [3]:
import pandas as pd
import pickle

# Load the CSV file
df_csv = pd.read_csv('data/csv_row_counts_0827.csv')

# Load the saved dataframe artifact from jupyter lab (parquet file)
artifact_path = r'C:\Users\aeliz\Dropbox\Documents\Jupyter Notebooks\chap2_usgs\artifacts\run_20250829_115143\df.parquet'
df_summary = pd.read_parquet(artifact_path)
print("Successfully loaded artifact as parquet file")

print("Original dataframes info:")
print(f"CSV dataframe shape: {df_csv.shape}")
print(f"Summary dataframe shape: {df_summary.shape}")
print()

# STEP 1: Data integrity checks
print("=== DATA INTEGRITY CHECKS ===")
print(f"Original CSV total rows: {df_csv['number_rows'].sum():,}")
print(f"Original CSV shape: {df_csv.shape}")
print(f"Unique doc_id + page_number combinations in CSV: {df_csv[['doc_id', 'page_number']].drop_duplicates().shape[0]}")
print(f"Unique doc_id values in CSV: {df_csv['doc_id'].nunique()}")
print()

print(f"Summary dataframe shape: {df_summary.shape}")

# Create id_page_number if it doesn't exist
if 'id_page_number' not in df_summary.columns:
    df_summary['id_page_number'] = df_summary['id'].astype(str) + '_' + df_summary['page_number'].astype(str)
    print("Created id_page_number column")

print(f"Unique id_page_number combinations in summary: {df_summary['id_page_number'].nunique()}")
print(f"Unique id values in summary: {df_summary['id'].nunique()}")
print()

# Check for duplicates in CSV
csv_duplicates = df_csv.duplicated(subset=['doc_id', 'page_number', 'table_number']).sum()
print(f"Duplicate rows in CSV (same doc_id + page_number + table_number): {csv_duplicates}")

# Check for duplicates in summary using id_page_number
summary_duplicates = df_summary.duplicated(subset=['id_page_number']).sum()
print(f"Duplicate rows in summary (same id_page_number): {summary_duplicates}")
print()

# STEP 2: Handle duplicates in summary dataframe using id_page_number
print("=== HANDLING DUPLICATES ===")
print(f"Found {summary_duplicates:,} duplicate rows in summary dataframe based on id_page_number")

if summary_duplicates > 0:
    print("Deduplicating summary dataframe by id_page_number...")
    
    # Before deduplication, let's see what the duplicates look like
    duplicate_examples = df_summary[df_summary.duplicated(subset=['id_page_number'], keep=False)].sort_values(['id_page_number'])
    print("Sample duplicate entries:")
    print(duplicate_examples[['id_page_number', 'id', 'page_number', 'water_type_clean']].head(10))
    print()
    
    # Check if duplicates have the same water_type_clean classification
    duplicate_groups = duplicate_examples.groupby(['id_page_number'])['water_type_clean'].nunique()
    conflicting_classifications = duplicate_groups[duplicate_groups > 1].count()
    print(f"Number of id_page_number combinations with conflicting water type classifications: {conflicting_classifications}")
    print()

# Remove duplicates - keep first occurrence for each unique id_page_number
df_summary_clean = df_summary.drop_duplicates(subset=['id_page_number'], keep='first')
print(f"Summary dataframe after deduplication: {df_summary_clean.shape}")
print(f"Unique id_page_number combinations after cleaning: {df_summary_clean['id_page_number'].nunique()}")
print(f"Expected around 63,810 unique IDs - Actual: {df_summary_clean['id_page_number'].nunique():,}")
print()

# STEP 3: Aggregate CSV data by doc_id + page_number
print("=== AGGREGATION STEP ===")
df_csv_aggregated = df_csv.groupby(['doc_id', 'page_number']).agg({
    'number_rows': 'sum',
    'table_number': 'count'
}).reset_index()

df_csv_aggregated.rename(columns={'table_number': 'num_tables_on_page'}, inplace=True)

print(f"After aggregation - Total rows: {df_csv_aggregated['number_rows'].sum():,}")
print(f"Aggregated CSV shape: {df_csv_aggregated.shape}")
print(f"Unique documents in aggregated CSV: {df_csv_aggregated['doc_id'].nunique()}")
print()

# Check if aggregation preserves total row count
original_total = df_csv['number_rows'].sum()
aggregated_total = df_csv_aggregated['number_rows'].sum()
if original_total == aggregated_total:
    print("✓ Aggregation preserves total row count")
else:
    print(f"⚠️  Aggregation mismatch: Original {original_total:,} vs Aggregated {aggregated_total:,}")
print()

# STEP 4: Create unique IDs and merge with clean data
print("=== MERGE PREPARATION WITH CLEAN DATA ===")
df_csv_aggregated['unique_id'] = df_csv_aggregated['doc_id'].astype(str) + '_' + df_csv_aggregated['page_number'].astype(str)
# Use id_page_number as unique_id for the summary data
df_summary_clean['unique_id'] = df_summary_clean['id_page_number']

# Check for duplicates in unique_ids (should be 0 now)
csv_uid_duplicates = df_csv_aggregated['unique_id'].duplicated().sum()
summary_uid_duplicates_clean = df_summary_clean['unique_id'].duplicated().sum()
print(f"Duplicate unique_ids in CSV: {csv_uid_duplicates}")
print(f"Duplicate unique_ids in cleaned summary: {summary_uid_duplicates_clean}")

# Check overlap with cleaned data
common_ids_clean = set(df_csv_aggregated['unique_id']) & set(df_summary_clean['unique_id'])
print(f"Common unique IDs between datasets (after cleaning): {len(common_ids_clean):,}")
print(f"CSV unique IDs: {len(df_csv_aggregated['unique_id'].unique()):,}")
print(f"Cleaned summary unique IDs: {len(df_summary_clean['unique_id'].unique()):,}")
print()

# STEP 5: Perform merge with cleaned data
print("=== MERGE RESULTS WITH CLEAN DATA ===")
df_merged = pd.merge(df_csv_aggregated, df_summary_clean, on='unique_id', how='inner', suffixes=('_csv', '_summary'))

print(f"Merged dataframe shape: {df_merged.shape}")
print(f"Total rows after merge: {df_merged['number_rows'].sum():,}")
print(f"Unique documents after merge: {df_merged['doc_id'].nunique()}")

# Check if merge preserves numbers with cleaned data
pre_merge_total = df_csv_aggregated['number_rows'].sum()
post_merge_total = df_merged['number_rows'].sum()
if pre_merge_total == post_merge_total:
    print("✓ Merge preserves total row count")
else:
    print(f"⚠️  Numbers still don't match: Pre-merge {pre_merge_total:,} vs Post-merge {post_merge_total:,}")
print()

# STEP 6: Final analysis - total measurements by water type
print("=== FINAL ANALYSIS ===")
water_type_column = 'water_type_clean'  # Water type classification column

if water_type_column in df_merged.columns:
    # Group by water type and sum the number of rows (measurements)
    water_type_summary = df_merged.groupby(water_type_column).agg({
        'number_rows': 'sum',
        'unique_id': 'count',
        'num_tables_on_page': 'sum',
        'doc_id': 'nunique'  # Count unique documents
    }).reset_index()
    
    water_type_summary.columns = [water_type_column, 'total_measurements', 'num_pages', 'total_tables', 'num_documents']
    water_type_summary = water_type_summary.sort_values('total_measurements', ascending=False)
    
    print("FINAL RESULT - Total measurements by water type:")
    print(water_type_summary)
    
    # Save the final summary
    water_type_summary.to_csv('water_type_measurements_summary.csv', index=False)
    print(f"\nFinal summary saved as 'water_type_measurements_summary.csv'")
    
    # Show totals
    print(f"\nGRAND TOTALS:")
    print(f"Total measurements across all water types: {water_type_summary['total_measurements'].sum():,}")
    print(f"Total pages: {water_type_summary['num_pages'].sum():,}")
    print(f"Total documents: {water_type_summary['num_documents'].sum():,}")
    
else:
    print(f"Column '{water_type_column}' not found. Available columns in merged data:")
    print(list(df_merged.columns))
    print("\nPlease update the 'water_type_column' variable with the correct column name.")

# Save the merged dataframe for reference
df_merged.to_csv('merged_dataframe.csv', index=False)
print("\nDetailed merged dataframe saved as 'merged_dataframe.csv'")

Successfully loaded artifact as parquet file
Original dataframes info:
CSV dataframe shape: (105920, 4)
Summary dataframe shape: (107314, 27)

=== DATA INTEGRITY CHECKS ===
Original CSV total rows: 2,666,019
Original CSV shape: (105920, 4)
Unique doc_id + page_number combinations in CSV: 67365
Unique doc_id values in CSV: 595

Summary dataframe shape: (107314, 27)
Unique id_page_number combinations in summary: 63810
Unique id values in summary: 581

Duplicate rows in CSV (same doc_id + page_number + table_number): 0
Duplicate rows in summary (same id_page_number): 43504

=== HANDLING DUPLICATES ===
Found 43,504 duplicate rows in summary dataframe based on id_page_number
Deduplicating summary dataframe by id_page_number...
Sample duplicate entries:
       id_page_number     id page_number  water_type_clean
20563         10060_4  10060           4       Groundwater
20562         10060_4  10060           4  Stream discharge
106782        10060_6  10060           6        Irrigation
106781

C:\Users\aeliz\AppData\Local\Temp\ipykernel_25608\3746621707.py:98: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_summary_clean['unique_id'] = df_summary_clean['id_page_number']


Cleaned summary unique IDs: 63,810

=== MERGE RESULTS WITH CLEAN DATA ===
Merged dataframe shape: (63524, 32)
Total rows after merge: 2,512,776
Unique documents after merge: 580
⚠️  Numbers still don't match: Pre-merge 2,666,019 vs Post-merge 2,512,776

=== FINAL ANALYSIS ===
FINAL RESULT - Total measurements by water type:
    water_type_clean  total_measurements  num_pages  total_tables  \
7   Stream discharge             2206942      53335         84634   
0        Groundwater              137875       4876          8548   
5          Reservoir               91816       2620          4012   
1         Irrigation               25157        803          1195   
6            Springs               17277        816           886   
4      Precipitation                8159        299           415   
8      Water quality                6940        203           248   
2  Not water related                6292        220           256   
3              Other                 128          3  

## with decade counts 

In [1]:
import pandas as pd
import pickle
import numpy as np

# Load the CSV file
df_csv = pd.read_csv('data/csv_row_counts_0827.csv')

# Load the saved dataframe artifact from jupyter lab (parquet file)
artifact_path = r'C:\Users\aeliz\Dropbox\Documents\Jupyter Notebooks\chap2_usgs\artifacts\run_20250829_115143\df.parquet'
df_summary = pd.read_parquet(artifact_path)
print("Successfully loaded artifact as parquet file")

print("Original dataframes info:")
print(f"CSV dataframe shape: {df_csv.shape}")
print(f"Summary dataframe shape: {df_summary.shape}")
print()

# STEP 1: Data integrity checks
print("=== DATA INTEGRITY CHECKS ===")
print(f"Original CSV total rows: {df_csv['number_rows'].sum():,}")
print(f"Original CSV shape: {df_csv.shape}")
print(f"Unique doc_id + page_number combinations in CSV: {df_csv[['doc_id', 'page_number']].drop_duplicates().shape[0]}")
print(f"Unique doc_id values in CSV: {df_csv['doc_id'].nunique()}")
print()

print(f"Summary dataframe shape: {df_summary.shape}")

# Create id_page_number if it doesn't exist
if 'id_page_number' not in df_summary.columns:
    df_summary['id_page_number'] = df_summary['id'].astype(str) + '_' + df_summary['page_number'].astype(str)
    print("Created id_page_number column")

print(f"Unique id_page_number combinations in summary: {df_summary['id_page_number'].nunique()}")
print(f"Unique id values in summary: {df_summary['id'].nunique()}")
print()

# Check for duplicates in CSV
csv_duplicates = df_csv.duplicated(subset=['doc_id', 'page_number', 'table_number']).sum()
print(f"Duplicate rows in CSV (same doc_id + page_number + table_number): {csv_duplicates}")

# Check for duplicates in summary using id_page_number
summary_duplicates = df_summary.duplicated(subset=['id_page_number']).sum()
print(f"Duplicate rows in summary (same id_page_number): {summary_duplicates}")
print()

# STEP 2: Handle duplicates in summary dataframe using id_page_number
print("=== HANDLING DUPLICATES ===")
print(f"Found {summary_duplicates:,} duplicate rows in summary dataframe based on id_page_number")

if summary_duplicates > 0:
    print("Deduplicating summary dataframe by id_page_number...")
    
    # Before deduplication, let's see what the duplicates look like
    duplicate_examples = df_summary[df_summary.duplicated(subset=['id_page_number'], keep=False)].sort_values(['id_page_number'])
    print("Sample duplicate entries:")
    print(duplicate_examples[['id_page_number', 'id', 'page_number', 'water_type_clean']].head(10))
    print()
    
    # Check if duplicates have the same water_type_clean classification
    duplicate_groups = duplicate_examples.groupby(['id_page_number'])['water_type_clean'].nunique()
    conflicting_classifications = duplicate_groups[duplicate_groups > 1].count()
    print(f"Number of id_page_number combinations with conflicting water type classifications: {conflicting_classifications}")
    print()

# Remove duplicates - keep first occurrence for each unique id_page_number
df_summary_clean = df_summary.drop_duplicates(subset=['id_page_number'], keep='first')
print(f"Summary dataframe after deduplication: {df_summary_clean.shape}")
print(f"Unique id_page_number combinations after cleaning: {df_summary_clean['id_page_number'].nunique()}")
print(f"Expected around 63,810 unique IDs - Actual: {df_summary_clean['id_page_number'].nunique():,}")
print()

# STEP 3: Aggregate CSV data by doc_id + page_number
print("=== AGGREGATION STEP ===")
df_csv_aggregated = df_csv.groupby(['doc_id', 'page_number']).agg({
    'number_rows': 'sum',
    'table_number': 'count'
}).reset_index()

df_csv_aggregated.rename(columns={'table_number': 'num_tables_on_page'}, inplace=True)

print(f"After aggregation - Total rows: {df_csv_aggregated['number_rows'].sum():,}")
print(f"Aggregated CSV shape: {df_csv_aggregated.shape}")
print(f"Unique documents in aggregated CSV: {df_csv_aggregated['doc_id'].nunique()}")
print()

# Check if aggregation preserves total row count
original_total = df_csv['number_rows'].sum()
aggregated_total = df_csv_aggregated['number_rows'].sum()
if original_total == aggregated_total:
    print("✓ Aggregation preserves total row count")
else:
    print(f"⚠️  Aggregation mismatch: Original {original_total:,} vs Aggregated {aggregated_total:,}")
print()

# STEP 4: Create unique IDs and merge with clean data
print("=== MERGE PREPARATION WITH CLEAN DATA ===")
df_csv_aggregated['unique_id'] = df_csv_aggregated['doc_id'].astype(str) + '_' + df_csv_aggregated['page_number'].astype(str)
# Use id_page_number as unique_id for the summary data
df_summary_clean['unique_id'] = df_summary_clean['id_page_number']

# Check for duplicates in unique_ids (should be 0 now)
csv_uid_duplicates = df_csv_aggregated['unique_id'].duplicated().sum()
summary_uid_duplicates_clean = df_summary_clean['unique_id'].duplicated().sum()
print(f"Duplicate unique_ids in CSV: {csv_uid_duplicates}")
print(f"Duplicate unique_ids in cleaned summary: {summary_uid_duplicates_clean}")

# Check overlap with cleaned data
common_ids_clean = set(df_csv_aggregated['unique_id']) & set(df_summary_clean['unique_id'])
print(f"Common unique IDs between datasets (after cleaning): {len(common_ids_clean):,}")
print(f"CSV unique IDs: {len(df_csv_aggregated['unique_id'].unique()):,}")
print(f"Cleaned summary unique IDs: {len(df_summary_clean['unique_id'].unique()):,}")
print()

# STEP 5: Perform merge with cleaned data
print("=== MERGE RESULTS WITH CLEAN DATA ===")
df_merged = pd.merge(df_csv_aggregated, df_summary_clean, on='unique_id', how='inner', suffixes=('_csv', '_summary'))

print(f"Merged dataframe shape: {df_merged.shape}")
print(f"Total rows after merge: {df_merged['number_rows'].sum():,}")
print(f"Unique documents after merge: {df_merged['doc_id'].nunique()}")

# Check if merge preserves numbers with cleaned data
pre_merge_total = df_csv_aggregated['number_rows'].sum()
post_merge_total = df_merged['number_rows'].sum()
if pre_merge_total == post_merge_total:
    print("✓ Merge preserves total row count")
else:
    print(f"⚠️  Numbers still don't match: Pre-merge {pre_merge_total:,} vs Post-merge {post_merge_total:,}")
print()

# STEP 5.5: ADD DECADE CALCULATION
print("=== DECADE CALCULATION ===")

# --- Helper function to calculate decades ---
def calculate_decade(year):
    """Calculate decade from year (e.g., 1823 -> 1820)"""
    if pd.isna(year):
        return np.nan
    try:
        year_int = int(float(year))
        return (year_int // 10) * 10
    except (ValueError, TypeError):
        return np.nan

# Check if year columns exist
year_columns = ['year_start', 'year_end']
missing_columns = [col for col in year_columns if col not in df_merged.columns]

if missing_columns:
    print(f"⚠️  Missing year columns: {missing_columns}")
    print(f"Available columns: {list(df_merged.columns)}")
    print("Cannot perform decade analysis without year data.")
else:
    # Add decade columns for both year_start and year_end
    df_merged['decade_start'] = df_merged['year_start'].apply(calculate_decade)
    df_merged['decade_end'] = df_merged['year_end'].apply(calculate_decade)
    
    print("✓ Decade columns added successfully")
    
    # Show decade distribution for start years
    print(f"Unique start decades: {sorted(df_merged['decade_start'].dropna().unique().astype(int))}")
    print(f"Unique end decades: {sorted(df_merged['decade_end'].dropna().unique().astype(int))}")
    
    # Show decade distribution
    decade_start_counts = df_merged['decade_start'].value_counts().sort_index()
    decade_end_counts = df_merged['decade_end'].value_counts().sort_index()
    
    print("\nStart decade distribution (number of pages):")
    for decade, count in decade_start_counts.items():
        if not pd.isna(decade):
            print(f"  {int(decade)}s: {count:,} pages")
    
    print("\nEnd decade distribution (number of pages):")
    for decade, count in decade_end_counts.items():
        if not pd.isna(decade):
            print(f"  {int(decade)}s: {count:,} pages")
    print()

# STEP 6: Final analysis - total measurements by water type
print("=== FINAL ANALYSIS BY WATER TYPE ===")
water_type_column = 'water_type_clean'  # Water type classification column

if water_type_column in df_merged.columns:
    # Group by water type and sum the number of rows (measurements)
    water_type_summary = df_merged.groupby(water_type_column).agg({
        'number_rows': 'sum',
        'unique_id': 'count',
        'num_tables_on_page': 'sum',
        'doc_id': 'nunique'  # Count unique documents
    }).reset_index()
    
    water_type_summary.columns = [water_type_column, 'total_measurements', 'num_pages', 'total_tables', 'num_documents']
    water_type_summary = water_type_summary.sort_values('total_measurements', ascending=False)
    
    print("FINAL RESULT - Total measurements by water type:")
    print(water_type_summary)
    
    # Save the final summary
    water_type_summary.to_csv('water_type_measurements_summary.csv', index=False)
    print(f"\nFinal summary saved as 'water_type_measurements_summary.csv'")
    
    # Show totals
    print(f"\nGRAND TOTALS:")
    print(f"Total measurements across all water types: {water_type_summary['total_measurements'].sum():,}")
    print(f"Total pages: {water_type_summary['num_pages'].sum():,}")
    print(f"Total documents: {water_type_summary['num_documents'].sum():,}")
    
else:
    print(f"Column '{water_type_column}' not found. Available columns in merged data:")
    print(list(df_merged.columns))
    print("\nPlease update the 'water_type_column' variable with the correct column name.")

# STEP 7: DECADE ANALYSIS
if 'decade_start' in df_merged.columns and 'decade_end' in df_merged.columns:
    print("\n=== DECADE ANALYSIS ===")
    
    # Total measurements by start decade
    decade_start_summary = df_merged.dropna(subset=['decade_start']).groupby('decade_start').agg({
        'number_rows': 'sum',
        'unique_id': 'count',
        'num_tables_on_page': 'sum',
        'doc_id': 'nunique'
    }).reset_index()
    
    decade_start_summary.columns = ['decade_start', 'total_measurements', 'num_pages', 'total_tables', 'num_documents']
    decade_start_summary = decade_start_summary.sort_values('decade_start')
    decade_start_summary['decade_label'] = decade_start_summary['decade_start'].astype(int).astype(str) + 's'
    
    print("Total measurements by start decade:")
    print(decade_start_summary[['decade_label', 'total_measurements', 'num_pages', 'total_tables', 'num_documents']])
    print()
    
    # Total measurements by end decade
    decade_end_summary = df_merged.dropna(subset=['decade_end']).groupby('decade_end').agg({
        'number_rows': 'sum',
        'unique_id': 'count',
        'num_tables_on_page': 'sum',
        'doc_id': 'nunique'
    }).reset_index()
    
    decade_end_summary.columns = ['decade_end', 'total_measurements', 'num_pages', 'total_tables', 'num_documents']
    decade_end_summary = decade_end_summary.sort_values('decade_end')
    decade_end_summary['decade_label'] = decade_end_summary['decade_end'].astype(int).astype(str) + 's'
    
    print("Total measurements by end decade:")
    print(decade_end_summary[['decade_label', 'total_measurements', 'num_pages', 'total_tables', 'num_documents']])
    print()
    
    # Water type by decade analysis (using start decade)
    if water_type_column in df_merged.columns:
        print("=== WATER TYPE BY START DECADE ANALYSIS ===")
        decade_water_type_start = df_merged.dropna(subset=['decade_start']).groupby(['decade_start', water_type_column]).agg({
            'number_rows': 'sum',
            'unique_id': 'count'
        }).reset_index()
        
        decade_water_type_start.columns = ['decade_start', water_type_column, 'total_measurements', 'num_pages']
        decade_water_type_start = decade_water_type_start.sort_values(['decade_start', 'total_measurements'], ascending=[True, False])
        decade_water_type_start['decade_label'] = decade_water_type_start['decade_start'].astype(int).astype(str) + 's'
        
        print("Measurements by start decade and water type:")
        print(decade_water_type_start[['decade_label', water_type_column, 'total_measurements', 'num_pages']])
        
        # Create pivot table for better visualization
        pivot_decade_water_start = decade_water_type_start.pivot(index='decade_start', columns=water_type_column, values='total_measurements').fillna(0)
        pivot_decade_water_start.index = pivot_decade_water_start.index.astype(int).astype(str) + 's'
        
        print(f"\nPivot table - Measurements by start decade and water type:")
        print(pivot_decade_water_start)
        
        # Water type by end decade analysis
        print("\n=== WATER TYPE BY END DECADE ANALYSIS ===")
        decade_water_type_end = df_merged.dropna(subset=['decade_end']).groupby(['decade_end', water_type_column]).agg({
            'number_rows': 'sum',
            'unique_id': 'count'
        }).reset_index()
        
        decade_water_type_end.columns = ['decade_end', water_type_column, 'total_measurements', 'num_pages']
        decade_water_type_end = decade_water_type_end.sort_values(['decade_end', 'total_measurements'], ascending=[True, False])
        decade_water_type_end['decade_label'] = decade_water_type_end['decade_end'].astype(int).astype(str) + 's'
        
        print("Measurements by end decade and water type:")
        print(decade_water_type_end[['decade_label', water_type_column, 'total_measurements', 'num_pages']])
        
        # Create pivot table for end decades
        pivot_decade_water_end = decade_water_type_end.pivot(index='decade_end', columns=water_type_column, values='total_measurements').fillna(0)
        pivot_decade_water_end.index = pivot_decade_water_end.index.astype(int).astype(str) + 's'
        
        print(f"\nPivot table - Measurements by end decade and water type:")
        print(pivot_decade_water_end)
        
        # Save decade analyses
        decade_start_summary.to_csv('decade_start_measurements_summary.csv', index=False)
        decade_end_summary.to_csv('decade_end_measurements_summary.csv', index=False)
        decade_water_type_start.to_csv('decade_start_water_type_measurements.csv', index=False)
        decade_water_type_end.to_csv('decade_end_water_type_measurements.csv', index=False)
        pivot_decade_water_start.to_csv('decade_start_water_type_pivot.csv')
        pivot_decade_water_end.to_csv('decade_end_water_type_pivot.csv')
        
        print(f"\nDecade analyses saved:")
        print(f"  - 'decade_start_measurements_summary.csv' (totals by start decade)")
        print(f"  - 'decade_end_measurements_summary.csv' (totals by end decade)")
        print(f"  - 'decade_start_water_type_measurements.csv' (start decade breakdown)")
        print(f"  - 'decade_end_water_type_measurements.csv' (end decade breakdown)")
        print(f"  - 'decade_start_water_type_pivot.csv' (start decade pivot table)")
        print(f"  - 'decade_end_water_type_pivot.csv' (end decade pivot table)")

# Save the merged dataframe for reference
df_merged.to_csv('merged_dataframe.csv', index=False)
print("\nDetailed merged dataframe saved as 'merged_dataframe.csv'")

Successfully loaded artifact as parquet file
Original dataframes info:
CSV dataframe shape: (105920, 4)
Summary dataframe shape: (107314, 27)

=== DATA INTEGRITY CHECKS ===
Original CSV total rows: 2,666,019
Original CSV shape: (105920, 4)
Unique doc_id + page_number combinations in CSV: 67365
Unique doc_id values in CSV: 595

Summary dataframe shape: (107314, 27)
Unique id_page_number combinations in summary: 63810
Unique id values in summary: 581

Duplicate rows in CSV (same doc_id + page_number + table_number): 0
Duplicate rows in summary (same id_page_number): 43504

=== HANDLING DUPLICATES ===
Found 43,504 duplicate rows in summary dataframe based on id_page_number
Deduplicating summary dataframe by id_page_number...
Sample duplicate entries:
       id_page_number     id page_number  water_type_clean
20563         10060_4  10060           4       Groundwater
20562         10060_4  10060           4  Stream discharge
106782        10060_6  10060           6        Irrigation
106781

C:\Users\aeliz\AppData\Local\Temp\ipykernel_9308\1953127848.py:99: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_summary_clean['unique_id'] = df_summary_clean['id_page_number']


Duplicate unique_ids in CSV: 0
Duplicate unique_ids in cleaned summary: 0
Common unique IDs between datasets (after cleaning): 63,524
CSV unique IDs: 67,365
Cleaned summary unique IDs: 63,810

=== MERGE RESULTS WITH CLEAN DATA ===
Merged dataframe shape: (63524, 32)
Total rows after merge: 2,512,776
Unique documents after merge: 580
⚠️  Numbers still don't match: Pre-merge 2,666,019 vs Post-merge 2,512,776

=== DECADE CALCULATION ===
✓ Decade columns added successfully
Unique start decades: [np.int64(1800), np.int64(1810), np.int64(1840), np.int64(1850), np.int64(1860), np.int64(1870), np.int64(1880), np.int64(1890), np.int64(1900), np.int64(1910), np.int64(1920), np.int64(1930), np.int64(1940), np.int64(1950), np.int64(1960), np.int64(1970), np.int64(1980)]
Unique end decades: [np.int64(1810), np.int64(1860), np.int64(1870), np.int64(1880), np.int64(1890), np.int64(1900), np.int64(1910), np.int64(1920), np.int64(1930), np.int64(1940), np.int64(1950), np.int64(1960), np.int64(1970), np